In [11]:
from conllu import parse_incr

In [7]:
import stanza
stanza.download("fr") #download french model
nlp = stanza.Pipeline("fr") #initialize French neural pipeline
doc = nlp("Le chat dort sur le canapé.")

2026-10-22 16:14:33 INFO: Downloaded file to /home/mace/.cache/stanza/1.15.0/resources/resources.json
2026-10-22 16:14:33 INFO: Downloading default packages for language: fr (French) ...
2026-10-22 16:14:33 INFO: File exists: /home/mace/.cache/stanza/1.15.0/resources/fr/default.zip
2026-10-22 16:14:35 INFO: Finished downloading models and saved to /home/mace/.cache/stanza/1.15.0/resources
2026-10-22 16:14:35 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-10-22 16:14:35 INFO: Downloaded file to /home/mace/.cache/stanza/1.15.0/resources/resources.json
2026-10-22 16:14:36 INFO: Loading these models for language: fr (French):
| Processor | Package            |
----------------------------------
| tokenize  | combined_nocharlm  |
| mwt       | combined           |
| pos       | combined_charlm    |
| lemma     | combined_nocharlm  |
| deppar

Available processors:
|num| Processor | Package            |
|--|-----------|--------------------|
|1| tokenize  | combined_nocharlm  | 
|2| mwt       | combined           |
|3| pos       | combined_charlm    |
|4| lemma     | combined_nocharlm  |
|5| depparse  | combined_charlm    |
|6|ner       | wikinergold_charlm |

1. Tokenizes the text and performs sentence segmentation.
2. Expands multi-word tokens (MWT) predicted by the TokenizeProcessor.
3. Labels tokens with their universal POS (UPOS) tags, treebank-specific POS (XPOS) tags, and universal morphological features (UFeats).
4. Generates the word lemmas for all words in the Document.
5. Provides an accurate syntactic dependency parsing analysis.

In [8]:
#Le chat dort sur le canapé.
print(doc)

[
  [
    {
      "id": 1,
      "text": "Le",
      "lemma": "le",
      "upos": "DET",
      "feats": "Definite=Def|Gender=Masc|Number=Sing|PronType=Art",
      "head": 2,
      "deprel": "det",
      "start_char": 0,
      "end_char": 2,
      "ner": "O",
      "multi_ner": [
        "O"
      ]
    },
    {
      "id": 2,
      "text": "chat",
      "lemma": "chat",
      "upos": "NOUN",
      "feats": "Gender=Masc|Number=Sing",
      "head": 3,
      "deprel": "nsubj",
      "start_char": 3,
      "end_char": 7,
      "ner": "O",
      "multi_ner": [
        "O"
      ]
    },
    {
      "id": 3,
      "text": "dort",
      "lemma": "dormir",
      "upos": "VERB",
      "feats": "Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin",
      "head": 0,
      "deprel": "root",
      "start_char": 8,
      "end_char": 12,
      "ner": "O",
      "multi_ner": [
        "O"
      ]
    },
    {
      "id": 4,
      "text": "sur",
      "lemma": "sur",
      "upos": "ADP",
      "head"

entities
tokens(workds) and sentences : .tokens, .sentences
text: .text

In [9]:
from collections import defaultdict, Counter

In [12]:
nlp = stanza.Pipeline("fr", processors="tokenize,mwt,pos,lemma,depparse")

# 1) Build an inflection lexicon from the treebank
lex = defaultdict(Counter)   # (lemma, upos) -> Counter{(feats, form): count}
with open("fr_gsd-ud-train.conllu", encoding="utf-8") as f:
    for sent in parse_incr(f):
        for t in sent:
            if not isinstance(t["id"], int):      # skip multiword ranges (du, au...)
                continue
            feats = tuple(sorted((t["feats"] or {}).items()))
            lex[(t["lemma"], t["upos"])][(feats, t["form"])] += 1

2026-10-22 16:15:30 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-10-22 16:15:30 INFO: Downloaded file to /home/mace/.cache/stanza/1.15.0/resources/resources.json
2026-10-22 16:15:30 INFO: Loading these models for language: fr (French):
| Processor | Package           |
---------------------------------
| tokenize  | combined_nocharlm |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |
| depparse  | combined_charlm   |

2026-10-22 16:15:30 INFO: Using device: cuda
2026-10-22 16:15:30 INFO: Loading: tokenize
2026-10-22 16:15:30 INFO: Loading: mwt
2026-10-22 16:15:30 INFO: Loading: pos
2026-10-22 16:15:32 INFO: Loading: lemma
2026-10-22 16:15:32 INFO: Loading: depparse
2026-10-22 16:15:32 INFO: Done loading processors!


In [ ]:
def inflect(lemma, upos, target):
    """Pick the known form whose features best match `target` (Number must match)."""
    best, best_key = None, (-1, -1)
    for (items, form), count in lex.get((lemma, upos), {}).items():
        d = dict(items)
        if d.get("Number") != target.get("Number"):
            continue
        score = sum(d.get(k) == v for k, v in target.items())
        if (score, count) > best_key:
            best, best_key = form, (score, count)
    return best

def feats(w):
    return dict(p.split("=") for p in w.feats.split("|")) if w.feats else {}

def match_case(src, new):
    return new.capitalize() if src[:1].isupper() else new

# 2) Rules
def check(text):
    doc, issues = nlp(text), []
    for sent in doc.sentences:
        words = sent.words
        for w in words:
            if w.head == 0:
                continue
            head, wf, hf = words[w.head - 1], feats(w), feats(words[w.head - 1])

            # Rule A: determiner / adjective must agree with its noun
            if w.deprel in ("det", "amod") and head.upos in ("NOUN", "PROPN"):
                target = dict(wf)
                for k in ("Number", "Gender"):
                    if k in hf and k in wf and hf[k] != wf[k]:
                        target[k] = hf[k]
                if target != wf:
                    fix = inflect(w.lemma, w.upos, target)
                    if fix and fix.lower() != w.text.lower():
                        issues.append((w.text, match_case(w.text, fix),
                                       f"{w.text} should agree with '{head.text}'"))

            # Rule B: verb must agree with its subject
            if w.deprel == "nsubj" and head.upos in ("VERB", "AUX"):
                if any(x.head == w.id and x.deprel == "conj" for x in words):
                    continue                      # "Marie et Paul" -> plural is fine
                vf = feats(head)
                target = dict(vf)
                if "Number" in wf and wf["Number"] != vf.get("Number"):
                    target["Number"] = wf["Number"]
                    if w.upos == "PRON" and "Person" in wf:
                        target["Person"] = wf["Person"]
                if target != vf and vf.get("VerbForm") == "Fin":
                    fix = inflect(head.lemma, head.upos, target)
                    if fix and fix.lower() != head.text.lower():
                        issues.append((head.text, fix,
                                       f"'{head.text}' should agree with subject '{w.text}'"))
    return issues


In [44]:
for issue in check("Les chats noir dorment sur la canapé. Il mangent beaucoup."):
    print(issue)

(10, 14, 'noirs', "'noir' should agree with 'chats'", None)
(27, 29, 'le', "'la' should agree with 'canapé'", None)
(41, 48, 'mange', "'mangent' should agree with subject 'Il'", (38, 40, 'Ils'))


Apply suggestion to the text

In [ ]:
def span(w):
    """Character offsets of a word in the original text (None for contracted words like du/au)."""
    tok = w.parent
    if len(tok.words) > 1:
        return None
    return tok.start_char, tok.end_char


In [ ]:
def check(text):
    doc, issues = nlp(text), []
    for sent in doc.sentences:
        words = sent.words
        for w in words:
            if w.head == 0:
                continue
            head, wf, hf = words[w.head - 1], feats(w), feats(words[w.head - 1])

            # Rule A: determiner / adjective vs noun
            if w.deprel in ("det", "amod") and head.upos in ("NOUN", "PROPN"):
                target = dict(wf)
                for k in ("Number", "Gender"):
                    if k in hf and k in wf and hf[k] != wf[k]:
                        target[k] = hf[k]
                if target != wf:
                    fix = inflect(w.lemma, w.upos, target)
                    sp = span(w)
                    if fix and sp and fix.lower() != w.text.lower():
                        issues.append((sp[0], sp[1], match_case(w.text, fix),
                                       f"'{w.text}' should agree with '{head.text}'", None))

            # Rule B: verb vs subject
            if w.deprel == "nsubj" and head.upos in ("VERB", "AUX"):
                if any(x.head == w.id and x.deprel == "conj" for x in words):
                    continue
                vf = feats(head)
                target = dict(vf)
                if "Number" in wf and wf["Number"] != vf.get("Number"):
                    target["Number"] = wf["Number"]
                    if w.upos == "PRON" and "Person" in wf:
                        target["Person"] = wf["Person"]
                if target != vf and vf.get("VerbForm") == "Fin":
                    verb_fix = inflect(head.lemma, head.upos, target)
                    vsp = span(head)

                    # Option 2: change the subject instead of the verb
                    subj_fix, ssp = None, span(w)
                    if w.upos == "PRON" and "Number" in vf:
                        subj_fix = inflect(w.lemma, w.upos, {**wf, "Number": vf["Number"]})

                    if verb_fix and vsp and verb_fix.lower() != head.text.lower():
                        alt = None
                        if subj_fix and ssp and subj_fix.lower() != w.text.lower():
                            alt = (ssp[0], ssp[1], match_case(w.text, subj_fix))
                        issues.append((vsp[0], vsp[1], verb_fix,
                                       f"'{head.text}' should agree with subject '{w.text}'", alt))
    return issues



In [40]:

def correct(text):
    # apply edits right-to-left so earlier offsets stay valid
    for start, end, fix, _, _ in sorted(check(text), key=lambda x: x[0], reverse=True):
        text = text[:start] + fix + text[end:]
    return text



In [43]:

text = "Les chats noir dorment sur la canapé. Il mangent beaucoup."
for start, end, fix, msg, alt in check(text):
    print(msg)
print()
print(correct(text))

'noir' should agree with 'chats'
'la' should agree with 'canapé'
'mangent' should agree with subject 'Il'

Les chats noirs dorment sur le canapé. Il mange beaucoup.


In [1]:
from conllu import parse_incr
from collections import Counter

counts = Counter()
with open("fr_gsd-ud-train.conllu", encoding="utf-8") as f:
    for sent in parse_incr(f):
        for t in sent:
            if isinstance(t["id"], int):      # skip multiword ranges (du, au...)
                counts[t["upos"]] += 1

for upos, n in counts.most_common():
    print(f"{upos:6} {n}")

NOUN   66668
ADP    56336
DET    54113
PUNCT  39017
VERB   28185
PROPN  24656
ADJ    21000
PRON   15598
ADV    12760
AUX    11586
CCONJ  9436
NUM    9289
X      2712
SCONJ  2602
SYM    617
INTJ   72


In [2]:
from stanza.utils.conll18_ud_eval import load_conllu_file, evaluate
# run the pipeline over the test text, write predictions to a .conllu file, then:
gold = load_conllu_file("fr_gsd-ud-test.conllu")
pred = load_conllu_file("predictions.conllu")
scores = evaluate(gold, pred)
print({k: round(v.f1, 3) for k, v in scores.items() if k in ("UPOS", "Lemmas", "UAS", "LAS")})

/home/mace/miniconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'stanza.utils.conll18_ud_eval'